# Private-Credit Explorer — from one Call Report field to a market view

**The question:** how exposed are US banks to private credit, and can an outsider
measure it from free public data? **Yes** — and it all starts from a single field on
a regulatory filing.

The notebook runs in three stages (Shift+Enter through them):

1. **Load** — pull the raw data from the FDIC's public API and understand what it is.
2. **Aggregate (market-wide)** — size up the whole market: how big, how fast, how concentrated.
3. **Company-specific** — drill into individual banks: exposure relative to their own capital.

That **aggregate → company** arc is exactly how the article is structured, and how you'd
assess any market: scope the industry first, then underwrite the name.

*Requires internet for the Stage-1 API calls.*

---
## Stage 1 — Load the data

Everything starts here: the raw material and where it comes from. Nothing is analyzed yet —
we're just pulling numbers off a regulatory filing and stacking them into a table.

## The data source

**FDIC BankFind Suite — Financials API** (`https://api.fdic.gov/banks/financials`), free,
no key. It serves the quarterly **Call Report** (regulatory filing FFIEC 031/041) for every
FDIC-insured bank, as JSON.

**Reporting basis:** the *bank* (insured depository institution) level — each chartered bank
files its own report. This is **not** the holding-company level (that is a separate filing,
the FR Y-9C). We rely on this basis throughout, and later roll banks up to their parent by
name.

The call below requests one quarter. Each institution comes back as a record; here is one
bank's relevant fields, raw from the API.

In [ ]:
import json, urllib.parse, urllib.request

API = "https://api.fdic.gov/banks/financials"
FIELDS = ["CERT", "NAME", "NAMEHCR", "REPDTE",
          "ASSET", "LNLSNET", "LNNDEPC", "RBCT1J"]

def fdic_pull(repdte):
    query = urllib.parse.urlencode({
        "filters": f"REPDTE:{repdte}",
        "fields": ",".join(FIELDS),
        "limit": 10000, "format": "json",
    })
    with urllib.request.urlopen(f"{API}?{query}", timeout=90) as resp:
        return [d["data"] for d in json.load(resp)["data"]]

records = fdic_pull("20260331")           # 2026-Q1
print(f"{len(records)} institutions returned for 2026-Q1")

# one bank's raw record (Truist Bank, FDIC cert 9846)
next(r for r in records if r.get("CERT") == 9846)

## The fields, and what each one is

All dollar figures are in **thousands** ($000s) as filed. Field codes are the FDIC's.

| Field | What it is | From the Call Report |
|---|---|---|
| `CERT` | FDIC certificate no. — unique bank ID | (identifier) |
| `NAME` | Bank's legal name | (identifier) |
| `NAMEHCR` | Name of the top-tier **holding company** | (identifier) |
| `REPDTE` | Reporting date (quarter-end) | (identifier) |
| `ASSET` | Total assets | Schedule RC |
| `LNLSNET` | Net loans and leases | Schedule RC-C |
| `LNNDEPC` | **Loans to non-depository financial institutions (NDFI)** — the private-credit proxy | Schedule RC-C |
| `RBCT1J` | **Tier 1 (core) capital** — the loss-absorbing base | Schedule RC-R |

`LNNDEPC` is a newer line item: banks over $10B in assets have reported it only since
**2024-Q4**, which is why the panel starts there.

## Build the panel

Pull every quarter since the NDFI line began, keep only banks that actually report it
(the >$10B filers), and stack into one table. Dollar columns are converted to numbers.

This is exactly `analysis/pull_call_reports.py`, which also writes the result to
`analysis/outputs/call_report_ndfi_panel.parquet` (the file the article reads).

In [ ]:
import pandas as pd

QUARTERS = ["20241231", "20250331", "20250630", "20250930", "20251231", "20260331"]

rows = []
for q in QUARTERS:
    recs = [r for r in fdic_pull(q) if r.get("LNNDEPC") not in (None, "")]
    print(f"  {q}: {len(recs)} banks report an NDFI balance")
    rows.extend(recs)

panel = pd.DataFrame(rows)
panel["quarter"] = pd.to_datetime(panel["REPDTE"].astype(str), format="%Y%m%d")
for col in ["ASSET", "LNLSNET", "LNNDEPC", "RBCT1J"]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

print(f"\npanel: {len(panel)} rows across {panel['quarter'].nunique()} quarters")
panel.head()

---
## Stage 2 — Aggregate analysis (the market, top-down)

Before judging any single bank, size up the whole market — the same way you'd scope an
industry before underwriting one borrower in it. Three questions, all answered from the one
field we just loaded:

- **How big?** — the total
- **How fast?** — the growth trajectory
- **How concentrated?** — a big-bank story, or spread out?

…and one honest caveat about a number this data *can't* show you directly (drawn vs. committed).

### First, isolate one quarter

The panel holds several quarters. For the cross-sectional questions (how big *now*, who's
concentrated *now*) we work with the most recent one. `cur` = the latest-quarter slice we
reuse throughout.

In [ ]:
latest = panel["quarter"].max()
cur = panel[panel["quarter"] == latest].copy()
print("latest quarter:", latest.date(), "| banks reporting NDFI:", len(cur))

view = cur[["NAME", "NAMEHCR", "LNNDEPC", "RBCT1J", "ASSET"]].copy()
view[["LNNDEPC","RBCT1J","ASSET"]] = (view[["LNNDEPC","RBCT1J","ASSET"]] / 1e6).round(1)
view.sort_values("LNNDEPC", ascending=False).head(8)

### How big is the market?

The headline is the simplest calculation there is: **sum the one field** across every
reporting bank. That's the "$1.5 trillion" figure.

In [ ]:
# The whole "$1.5 trillion" headline is a single .sum() across all reporting banks.
total_bn = cur["LNNDEPC"].sum() / 1e6                        # $thousands -> $bn
print(f"Total drawn NDFI, {latest.date()}:  ${total_bn:,.0f}B   (${total_bn/1000:.2f}T)")

### Growth trajectory

Sum `LNNDEPC` by quarter. The NDFI line only begins 2024-Q4, so the series is short — and
the earliest quarters are muddied by *reclassification* (loans moved **into** the NDFI
bucket from other categories), so early "growth" is partly an accounting artifact. Read the
later quarters for the genuine trend.

In [ ]:
by_q = (panel.groupby("quarter")
              .agg(ndfi=("LNNDEPC", "sum"), loans=("LNLSNET", "sum")))
by_q["ndfi_bn"] = by_q["ndfi"] / 1e6
by_q["loans_bn"] = by_q["loans"] / 1e6
by_q["ndfi_qoq"] = by_q["ndfi_bn"].pct_change()
by_q[["ndfi_bn", "ndfi_qoq"]].round(3)

### Is NDFI growing faster than the rest of the loan book?

Compare NDFI growth to total-loan (`LNLSNET`) growth across the panel. Whalen's headline is
NDFI growing ~5x *other loan categories*; we only pulled the total loan line, not every
category, so this is NDFI vs. the whole book — directional, not the exact 5x.

In [ ]:
first, last = by_q.iloc[0], by_q.iloc[-1]
ndfi_growth = last.ndfi_bn / first.ndfi_bn - 1
loan_growth = last.loans_bn / first.loans_bn - 1
print(f"NDFI:         {ndfi_growth:+.1%}  over the panel")
print(f"Total loans:  {loan_growth:+.1%}  over the panel")
print("(NDFI growth is inflated by early reclassification — treat as an upper bound)")

### Concentration — a big-bank story

Sum the latest quarter by size tier. The takeaway that lets an outsider skip the ~4,000
small banks entirely: the giants *are* the market.

In [ ]:
tot = cur["LNNDEPC"].sum()
top10 = cur.nlargest(10, "LNNDEPC")["LNNDEPC"].sum()
big = cur.loc[cur["ASSET"] >= 100_000_000, "LNNDEPC"].sum()  # >$100B banks
print(f"Reporting banks:   {len(cur)}")
print(f"Top-10 banks hold: {top10/tot:.0%} of all NDFI")
print(f"Banks > $100B:     {big/tot:.0%}")

### The number you can't see — drawn vs. committed

Everything above is **drawn**: `LNNDEPC` is an *outstanding balance* (Schedule RC-C). The
larger risk is the **committed** line — the undrawn amount a bank has promised, which
stressed funds can draw all at once (*exposure at default*, a Basel-I idea). That figure
lives on a different schedule (**RC-L**, unused commitments) and isn't a clean per-NDFI
field, so estimating it takes inference — which is why the widely-cited ~$3T EAD number is
an **estimate**, not a sum. Illustratively, at a blended utilization (drawn ÷ committed)
the drawn book implies:

In [ ]:
drawn_bn = cur["LNNDEPC"].sum() / 1e6
print(f"drawn (clean sum):          ${drawn_bn/1000:.2f}T")
for util in (0.56, 0.47):
    print(f"implied committed @ {util:.0%}:   ${drawn_bn/util/1000:.1f}T  (illustrative)")

### Beyond the reporting banks

Only banks over $10B report NDFI directly, so this panel is ~80 firms — but they are
essentially the whole thing: banks over $100B alone are ~98% of it, and the FDIC's
*Quarterly Banking Profile* aggregate (all ~4,400 banks) lands within a rounding error of
our sum. What this data can **never** see is the non-bank market itself — the ~$2T of loans
the funds hold *directly*, which lives in fund filings (BDC 10-Ks, Form PF), not Call
Reports. Three concentric numbers, each less verifiable than the last:

| Number | Meaning | Source |
| --- | --- | --- |
| **~$1.5T** | bank NDFI **drawn** | one Call Report field (this notebook) |
| **~$3T** | bank NDFI **committed** (EAD) | Call Report + a utilization/CCF assumption |
| **~$2T** | the private-credit **market** | fund filings — outside the banking data entirely |

The number you can trust most is the smallest one.

---
## Stage 3 — Company-specific analysis (the banks, bottom-up)

Same field, different cut. Now we go bank-by-bank. Two transformations turn the raw rows
into a per-bank risk metric, then we rank and plot. This is the "who's standing closest to
the mine" view — and where the article's two case studies (First Citizens, Truist) live.

### Transform 1 — why we must roll subsidiaries up

`LNNDEPC` is reported per *bank*, but a big firm often operates several banks. Morgan Stanley
files **two** separate Call Reports. Leave them apart and the firm shows up as two points,
each with a nonsense ratio. First, see the problem:

In [ ]:
ms = cur.loc[cur["NAMEHCR"].str.contains("MORGAN STANLEY"),
             ["NAME", "NAMEHCR", "LNNDEPC", "RBCT1J"]].copy()
ms[["LNNDEPC","RBCT1J"]] = (ms[["LNNDEPC","RBCT1J"]] / 1e6).round(1)
ms

### …then group by the parent company

Sum each firm's subsidiaries into one row (group by `NAMEHCR`, the holding company). Morgan
Stanley is now a single, correct data point.

In [ ]:
grp = (cur.assign(entity=cur["NAMEHCR"].str.strip().replace("", pd.NA).fillna(cur["NAME"]))
          .groupby("entity")
          .agg(ndfi=("LNNDEPC","sum"), tier1=("RBCT1J","sum"),
               assets=("ASSET","sum"), loans=("LNLSNET","sum"))
          .reset_index())

ms_holdco = grp[grp["entity"].str.contains("MORGAN STANLEY")]
(ms_holdco[["ndfi","tier1"]] / 1e6).round(1)

### Transform 2 — the concentration metric

Now the number that matters. As a credit analyst you already have the instinct: **exposure
only means something against the capital that absorbs losses.** $40B of NDFI is trivial for
JPMorgan and huge for a mid-size bank.

So divide NDFI by **Tier 1 capital**. Over 100% means the private-credit book exceeds the
bank's entire core capital cushion. This one column is what the ranking and the chart are
built on.

In [ ]:
grp["ndfi_to_tier1"] = grp["ndfi"] / grp["tier1"]
grp["ndfi_bn"] = grp["ndfi"] / 1e6

ranking = (grp[grp["ndfi_bn"] >= 10]
           .sort_values("ndfi_to_tier1", ascending=False)
           [["entity", "ndfi_bn", "ndfi_to_tier1"]]
           .head(15).reset_index(drop=True))
ranking["ndfi_bn"] = ranking["ndfi_bn"].round(1)
ranking["ndfi_to_tier1"] = (ranking["ndfi_to_tier1"]*100).round(0).astype(int).astype(str) + "%"
ranking

### The picture — scale vs. concentration

Dollars (horizontal) and concentration (vertical) are *different axes*. Plot both — bubble
size = total assets — and the punchline appears: the biggest **lenders** (megabanks, far
right) aren't the most **exposed** (highest up). A rough version of the article's centerpiece.

In [ ]:
import matplotlib.pyplot as plt

g = grp[grp["ndfi_bn"] >= 1].copy()

# Bubble AREA should scale with total assets. matplotlib's `s` is area in points^2,
# so scale off assets in $bn (assets are $thousands -> /1e6). The old /1e4 made the
# megabanks ~400,000 pts and they ate the whole chart.
g["bubble"] = (g["assets"] / 1e6) * 0.35        # ~$10B bank -> 4 pts;  ~$4T bank -> ~1400 pts

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(g["ndfi_bn"], g["ndfi_to_tier1"], s=g["bubble"],
           alpha=0.45, color="#8CA0B4", edgecolor="#61758A", linewidth=0.6)
ax.set_xscale("log")
ax.set_ylim(0, 2.15)
ax.axhline(1.0, ls="--", color="#B4AD9E", lw=1)   # 100% of Tier 1 capital
ax.set_xlabel("NDFI loans, $bn (log)")
ax.set_ylabel("NDFI / Tier 1 capital")
ax.set_title("Scale is not concentration")

# label a few notable banks with clean names
labels = {"GOLDMAN SACHS GROUP INC THE": "Goldman Sachs",
          "FIRST CITIZENS BANCSHARES INC": "First Citizens",
          "TRUIST FINANCIAL CORP": "Truist"}
for raw, nice in labels.items():
    row = g[g["entity"] == raw]
    if len(row):
        ax.annotate(nice, (row["ndfi_bn"].iloc[0], row["ndfi_to_tier1"].iloc[0]),
                    fontsize=9, fontweight="bold", ha="center",
                    xytext=(0, 9), textcoords="offset points")

plt.tight_layout()
plt.show()

### Reading vs. computing — the First Citizens case

Not every number comes from the Call Report. The **concentration metric** we just computed
(NDFI ÷ Tier 1) is regulatory data. But First Citizens' **composition** — 85% capital-call
lines vs. 15% other, and the loss allowance on each — is something the bank *chooses to
disclose* in its 10-K MD&A. `LNNDEPC` is a single aggregate number; the Call Report **cannot
split it**.

What you *can* do from the data is **cross-check**: confirm the 10-K's disclosed NDFI total
ties to the Call Report line.

In [ ]:
import json, urllib.request

# Locate First Citizens' ACTUAL latest 10-K on SEC EDGAR (CIK 798941) — programmatically.
_req = urllib.request.Request(
    "https://data.sec.gov/submissions/CIK0000798941.json",
    headers={"User-Agent": "research benuehlinger@gmail.com"})
_r = json.load(urllib.request.urlopen(_req, timeout=30))["filings"]["recent"]
for form, date, acc, doc in zip(_r["form"], _r["filingDate"], _r["accessionNumber"], _r["primaryDocument"]):
    if form == "10-K":
        tenk_url  = f"https://www.sec.gov/Archives/edgar/data/798941/{acc.replace('-','')}/{doc}"
        tenk_date = date
        break
print(f"First Citizens latest 10-K: filed {tenk_date}")
print(f"  {tenk_url}\n")

# Cross-check: Call Report NDFI (COMPUTED from the FDIC API) vs the total DISCLOSED in that 10-K.
fcnca = cur[cur["CERT"] == 11063].iloc[0]
call_report_ndfi = fcnca["LNNDEPC"] / 1e6      # $bn, bank-level — computed
disclosed_ndfi   = 38.8                         # $bn — transcribed from the 10-K MD&A / Q1-26 slide above
print(f"Call Report LNNDEPC (computed):   ${call_report_ndfi:.1f}B")
print(f"10-K disclosure (from filing ^):  ${disclosed_ndfi:.1f}B")
print(f"tie-out:                          {call_report_ndfi/disclosed_ndfi - 1:+.1%}  (bank vs holding-company basis)")

They tie. So the *total* is corroborated by the regulator's number — but the **split and
allowances below are read straight from the 10-K, not computed.** Flagged as such, because
the regulatory data stops at the total:

In [ ]:
# Composition & allowances — READ (transcribed) from the 10-K MD&A fetched above.
# These are NARRATIVE disclosures; no structured feed (XBRL, API) exists for them, so they
# must be hand-read from the filing — which is exactly why the article calls this the
# "read the disclosure" case. Provenance is the fetched 10-K URL.
print(f"Source: {tenk_url}\n        (MD&A, filed {tenk_date}; holding-company basis)\n")
composition = pd.DataFrame([
    {"slice": "Capital-call lines",          "share": "85%", "allowance": "0.09%",
     "secured_by": "LPs' contractual commitments"},
    {"slice": "Other NDFI / private credit", "share": "15%", "allowance": "1.06%",
     "secured_by": "fund NAV / portfolio — valuation-exposed"},
])
composition

The **12× allowance gap** (0.09% vs 1.06%) is the bank's own evidence that the big number is
mostly the *safe* kind of exposure. It's the **composition** — visible only because First
Citizens discloses it — that turns a scary headline into a defensible book. Truist (the
article's Part 3) doesn't disclose this, which is exactly why you have to *infer* its
exposure from the Call Report instead.

### Your turn — variations

Edit and re-run any of these:
- swap the denominator: `grp["ndfi"] / grp["loans"]` → NDFI as a share of the *total loan book*
- change `latest` (Stage 2) to an earlier quarter from `panel["quarter"].unique()` and re-run
- filter to one firm: `cur[cur["NAMEHCR"].str.contains("WELLS")]`

Nothing here touches the site — break things freely.

In [ ]:
# NDFI as a share of total loans (LNLSNET), by holding company
grp["ndfi_to_loans"] = grp["ndfi"] / grp["loans"]
(grp[grp["ndfi_bn"] >= 10]
   .sort_values("ndfi_to_loans", ascending=False)
   [["entity", "ndfi_bn", "ndfi_to_loans"]]
   .head(10))